# Introduccion

La compañía Megaline desea recomendar automáticamente uno de sus planes actuales (Smart o Ultra) a los usuarios que aún utilizan planes heredados. Para ello se desarrollarán modelos de machine learning capaces de predecir qué plan es más adecuado a partir del comportamiento mensual de los clientes.

El objetivo del proyecto es construir un modelo de clasificación con la mayor exactitud posible. El criterio de aceptación establecido es una exactitud mínima de 0.75 sobre el conjunto de prueba.

## Importar librerias necesarias

In [33]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score

## Cargar y leer preliminarmente los datos

In [24]:
df = pd.read_csv('/datasets/users_behavior.csv')

print(df.head())
print(df.info())

   calls  minutes  messages   mb_used  is_ultra
0   40.0   311.90      83.0  19915.42         0
1   85.0   516.75      56.0  22696.96         0
2   77.0   467.66      86.0  21060.45         0
3  106.0   745.53      81.0   8437.39         1
4   66.0   418.74       1.0  14502.75         0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB
None


Conclusión inicial

El conjunto de datos contiene información sobre el comportamiento mensual de los usuarios. La variable objetivo es is_ultra, donde:

- 1 = Ultra
- 0 = Smart

No se observan valores ausentes ni problemas evidentes en los tipos de datos.

## Preparación de datos

#### Separación de características y variable objetivo:

In [25]:
features = df.drop(['is_ultra'], axis=1)
target = df['is_ultra']

#### División en entrenamiento, validación y prueba:

In [26]:
features_train, features_temp, target_train, target_temp = train_test_split(
    features,
    target,
    test_size=0.4,
    random_state=12345
)

features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp,
    target_temp,
    test_size=0.5,
    random_state=12345
)

#### Verificacion del tamaño

In [27]:
print(features_train.shape) 
print(features_valid.shape) 
print(features_test.shape)

(1928, 4)
(643, 4)
(643, 4)


Los datos fueron divididos en:

- 60% entrenamiento
- 20% validación
- 20% prueba

Esta división permite entrenar los modelos, seleccionar hiperparámetros y realizar una evaluación final independiente.

### Entrenamiento del modelo Decision Tree

In [28]:
best_tree_model = None
best_tree_result = 0
best_depth = 0

for depth in range(1, 11):
    model = DecisionTreeClassifier(random_state=12345, max_depth=depth)
    model.fit(features_train, target_train)
    predictions_valid = model.predict(features_valid)
    result = accuracy_score(target_valid, predictions_valid)
    
    if result > best_tree_result:
        best_tree_model = model
        best_tree_result = result
        best_depth = depth

print("Mejor árbol:")
print("max_depth =", best_depth)
print("accuracy =", best_tree_result)

Mejor árbol:
max_depth = 3
accuracy = 0.7853810264385692


##### Resultados

###### Mejor profundidad:

3

###### Exactitud:

0.7853810264385692

### Entrenamiento del modelo Random Forest

In [29]:
best_forest_model = None
best_forest_result = 0
best_est = 0
best_depth = 0

for est in range(10, 101, 10):
    for depth in range(1, 11):
        model = RandomForestClassifier(
            random_state=12345,
            n_estimators=est,
            max_depth=depth
        )
        model.fit(features_train, target_train)
        predictions_valid = model.predict(features_valid)
        result = accuracy_score(target_valid, predictions_valid)
        
        if result > best_forest_result:
            best_forest_model = model
            best_forest_result = result
            best_est = est
            best_depth = depth

print("Mejor bosque:")
print("n_estimators =", best_est)
print("max_depth =", best_depth)
print("accuracy =", best_forest_result)

Mejor bosque:
n_estimators = 40
max_depth = 8
accuracy = 0.8087091757387247


#### Resultado

##### Mejor número de árboles:

40

##### Mejor profundidad:

8

##### Exactitud:

0.8087091757387247

### Entrenamiento del modelo Linear Regression

In [30]:
model_lr = LinearRegression()
model_lr.fit(features_train, target_train)

predictions = model_lr.predict(features_valid)

predictions_class = (predictions >= 0.5).astype(int)

accuracy_lr = accuracy_score(target_valid, predictions_class)

print(accuracy_lr)

0.7573872472783826


##### Resultado

La exactitud obtenida por este modelo fue de 0.7573872472783826

### Comparación de modelos



- Linear Regression	0.75
- Decision Tree	0.78
- Random Forest	0.8


### Conclusión

El modelo con mejor desempeño fue:

Random Forest

Por esta razón será utilizado para la evaluación final.

### Evaluación en el conjunto de prueba

In [31]:
test_model= RandomForestClassifier()
test_model.fit(features_valid,target_valid)
predictions_test = test_model.predict(features_test) 
test_accuracy = accuracy_score(target_test, predictions_test)
print(test_accuracy)

0.7931570762052877


#### Resultado

Accuracy en test: 0.793

### Prueba de cordura

In [32]:
most_frequent = target_train.mode()[0]

predictions = [most_frequent] * len(target_test)

accuracy = accuracy_score(target_test, predictions)

print(accuracy)

0.6842923794712286


Para la prueba de cordura se comparó el modelo final con una estrategia simple que predice siempre la clase más frecuente. La estrategia base obtuvo una exactitud de 0.68, mientras que el modelo final alcanzó 0.793. Esto indica que el modelo logró identificar patrones en los datos y supera ampliamente una predicción trivial.

## Conclusiones generales

Se entrenaron tres modelos distintos para recomendar planes de Megaline.

La regresión lineal obtuvo el rendimiento más bajo debido a que el problema corresponde a una tarea de clasificación. El árbol de decisión logró una mejora considerable al capturar relaciones no lineales entre las variables. Finalmente, el bosque aleatorio presentó el mejor desempeño gracias a la combinación de múltiples árboles de decisión.

El modelo seleccionado alcanzó una exactitud superior al umbral requerido de 0.75 en el conjunto de prueba, por lo que cumple satisfactoriamente los objetivos del proyecto.